# Public Pipeline: Wellbore Geology Prediction

Этот блокнот будем строить по этапам. Каждый этап — отдельный смысловой блок: сначала делаем надежную основу проекта, потом загрузчик данных, baseline, физические кандидаты, ML-модели и stacking.


## Kaggle configuration and dataset check

In [ ]:

from dataclasses import dataclass
from pathlib import Path

import pandas as pd


'''
This block builds the Kaggle-first project configuration and a lightweight dataset inventory.
It does not train models or read the full dataset into memory.
'''


@dataclass(frozen=True)
class ProjectConfig:
    project_dir: Path
    dataset_dir: Path
    train_dir: Path
    test_dir: Path
    sample_submission_path: Path
    work_dir: Path
    seed: int = 42
    n_splits: int = 5

    @classmethod
    def for_kaggle(cls) -> "ProjectConfig":
        kaggle_dataset_dir = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
        current_dir = Path.cwd().resolve()
        local_project_dir = current_dir if (current_dir / "datasets").exists() else current_dir.parent
        local_dataset_dir = local_project_dir / "datasets"

        if kaggle_dataset_dir.exists():
            project_dir = Path("/kaggle/working")
            dataset_dir = kaggle_dataset_dir
            work_dir = Path("/kaggle/working")
        else:
            project_dir = local_project_dir
            dataset_dir = local_dataset_dir
            work_dir = project_dir / "working"

        return cls(
            project_dir=project_dir,
            dataset_dir=dataset_dir,
            train_dir=dataset_dir / "train",
            test_dir=dataset_dir / "test",
            sample_submission_path=dataset_dir / "sample_submission.csv",
            work_dir=work_dir,
        )

    def validate(self) -> None:
        required_paths = [
            self.dataset_dir,
            self.train_dir,
            self.test_dir,
            self.sample_submission_path,
        ]
        missing_paths = [path for path in required_paths if not path.exists()]
        if missing_paths:
            readable = "\n".join(str(path) for path in missing_paths)
            raise FileNotFoundError(f"Missing required dataset paths:\n{readable}")
        self.work_dir.mkdir(parents=True, exist_ok=True)


class DatasetInventory:
    def __init__(self, config: ProjectConfig):
        self.config = config

    def horizontal_files(self, split: str) -> list[Path]:
        return sorted((self.config.dataset_dir / split).glob("*__horizontal_well.csv"))

    def typewell_files(self, split: str) -> list[Path]:
        return sorted((self.config.dataset_dir / split).glob("*__typewell.csv"))

    def png_files(self, split: str) -> list[Path]:
        return sorted((self.config.dataset_dir / split).glob("*.png"))

    def well_ids(self, split: str) -> set[str]:
        return {path.name.split("__")[0] for path in self.horizontal_files(split)}

    def overview(self) -> pd.DataFrame:
        rows = []
        for split in ["train", "test"]:
            horizontal_wells = self.well_ids(split)
            typewell_wells = {path.name.split("__")[0] for path in self.typewell_files(split)}
            rows.append({
                "split": split,
                "horizontal_files": len(self.horizontal_files(split)),
                "typewell_files": len(self.typewell_files(split)),
                "png_files": len(self.png_files(split)),
                "wells_with_both_csvs": len(horizontal_wells & typewell_wells),
                "horizontal_only_wells": len(horizontal_wells - typewell_wells),
                "typewell_only_wells": len(typewell_wells - horizontal_wells),
            })
        return pd.DataFrame(rows)

    def schema_examples(self) -> pd.DataFrame:
        rows = []
        for split in ["train", "test"]:
            examples = {
                "horizontal": self.horizontal_files(split),
                "typewell": self.typewell_files(split),
            }
            for file_type, files in examples.items():
                if not files:
                    continue
                frame = pd.read_csv(files[0], nrows=5)
                rows.append({
                    "split": split,
                    "file_type": file_type,
                    "example_file": files[0].name,
                    "columns": ", ".join(frame.columns),
                    "preview_rows": len(frame),
                })
        return pd.DataFrame(rows)

    def submission_overview(self) -> pd.DataFrame:
        sample_submission = pd.read_csv(self.config.sample_submission_path)
        well_ids = sample_submission["id"].str.split("_", n=1).str[0]
        return pd.DataFrame([{
            "rows": len(sample_submission),
            "columns": ", ".join(sample_submission.columns),
            "wells": well_ids.nunique(),
            "first_id": sample_submission["id"].iloc[0],
            "last_id": sample_submission["id"].iloc[-1],
        }])


config = ProjectConfig.for_kaggle()
config.validate()
inventory = DatasetInventory(config)

print(f"Project directory: {config.project_dir}")
print(f"Dataset directory: {config.dataset_dir}")
print(f"Working directory: {config.work_dir}")

display(inventory.overview())
display(inventory.schema_examples())
display(inventory.submission_overview())


## Well loader and sequence split

This stage creates the small OOP layer we will reuse everywhere else.

`WellLoader` loads one well from Kaggle CSV files. `WellData` exposes the two parts of a horizontal well:

- known prefix: rows where `TVT_input` is available
- prediction zone: rows where `TVT_input` is missing

For train wells, the prediction zone also contains the real `TVT`, so later we can build validation targets. For test wells, the same object gives the exact submission ids.


In [ ]:

@dataclass(frozen=True)
class WellData:
    """Container for one horizontal well and its matching typewell."""

    well_id: str
    split: str
    horizontal: pd.DataFrame
    typewell: pd.DataFrame

    @property
    def known_mask(self) -> pd.Series:
        return self.horizontal["TVT_input"].notna()

    @property
    def prediction_mask(self) -> pd.Series:
        return self.horizontal["TVT_input"].isna()

    @property
    def known_rows(self) -> pd.DataFrame:
        return self.horizontal.loc[self.known_mask].copy()

    @property
    def prediction_rows(self) -> pd.DataFrame:
        return self.horizontal.loc[self.prediction_mask].copy()

    @property
    def last_known_row(self) -> pd.Series:
        known = self.known_rows
        if known.empty:
            raise ValueError(f"Well {self.well_id} has no known TVT_input rows")
        return known.iloc[-1]

    def prediction_ids(self) -> list[str]:
        return [f"{self.well_id}_{row_index}" for row_index in self.prediction_rows.index]

    def summary(self) -> pd.DataFrame:
        has_target = "TVT" in self.horizontal.columns
        return pd.DataFrame([{
            "split": self.split,
            "well_id": self.well_id,
            "horizontal_rows": len(self.horizontal),
            "typewell_rows": len(self.typewell),
            "known_rows": int(self.known_mask.sum()),
            "prediction_rows": int(self.prediction_mask.sum()),
            "has_target": has_target,
            "last_known_tvt": float(self.last_known_row["TVT_input"]),
            "last_known_md": float(self.last_known_row["MD"]),
        }])


class WellLoader:
    """Loads horizontal well and typewell CSV files from the configured dataset directory."""

    def __init__(self, config: ProjectConfig):
        self.config = config

    def well_ids(self, split: str) -> list[str]:
        split_dir = self.config.dataset_dir / split
        return sorted(path.name.split("__")[0] for path in split_dir.glob("*__horizontal_well.csv"))

    def load(self, well_id: str, split: str) -> WellData:
        split_dir = self.config.dataset_dir / split
        horizontal_path = split_dir / f"{well_id}__horizontal_well.csv"
        typewell_path = split_dir / f"{well_id}__typewell.csv"
        if not horizontal_path.exists() or not typewell_path.exists():
            raise FileNotFoundError(f"Missing well files for {split}/{well_id}")
        return WellData(
            well_id=well_id,
            split=split,
            horizontal=pd.read_csv(horizontal_path),
            typewell=pd.read_csv(typewell_path),
        )

    def first(self, split: str) -> WellData:
        well_ids = self.well_ids(split)
        if not well_ids:
            raise ValueError(f"No wells found for split={split}")
        return self.load(well_ids[0], split)


def submission_alignment(loader: WellLoader, config: ProjectConfig) -> pd.DataFrame:
    """Checks that sample_submission ids match missing TVT_input rows in test wells."""

    sample_submission = pd.read_csv(config.sample_submission_path)
    expected_ids = []
    for well_id in loader.well_ids("test"):
        expected_ids.extend(loader.load(well_id, "test").prediction_ids())

    sample_ids = sample_submission["id"].astype(str).tolist()
    expected_set = set(expected_ids)
    sample_set = set(sample_ids)
    return pd.DataFrame([{
        "sample_rows": len(sample_ids),
        "expected_prediction_rows": len(expected_ids),
        "same_order": sample_ids == expected_ids,
        "missing_in_sample": len(expected_set - sample_set),
        "extra_in_sample": len(sample_set - expected_set),
    }])


loader = WellLoader(config)
train_example = loader.first("train")
test_example = loader.first("test")

display(train_example.summary())
display(test_example.summary())
display(submission_alignment(loader, config))


## Baseline predictors

Before adding ML, we need a simple reference score. This block adds two deterministic predictors:

- `LastKnownBaseline`: repeats the last available `TVT_input` across the prediction zone.
- `LinearSlopeBaseline`: estimates the recent TVT slope from the known prefix and continues it into the prediction zone.

These baselines are intentionally small. If later models do not beat them in GroupKFold validation, the model layer is not useful yet.


In [ ]:

import numpy as np


class LastKnownBaseline:
    """Repeats the last visible TVT value into the prediction zone."""

    name = "last_known"

    def predict(self, well: WellData) -> pd.Series:
        prediction_rows = well.prediction_rows
        last_tvt = float(well.last_known_row["TVT_input"])
        return pd.Series(last_tvt, index=prediction_rows.index, name=self.name)


class LinearSlopeBaseline:
    """Extends TVT from the known prefix using a recent median slope."""

    name = "linear_slope"

    def __init__(self, tail_rows: int = 80):
        self.tail_rows = tail_rows

    def _slope(self, known_rows: pd.DataFrame) -> float:
        tail = known_rows.tail(self.tail_rows)
        md_delta = tail["MD"].diff()
        tvt_delta = tail["TVT_input"].diff()
        valid = md_delta.gt(0) & tvt_delta.notna()
        if valid.sum() == 0:
            return 0.0
        return float((tvt_delta[valid] / md_delta[valid]).median())

    def predict(self, well: WellData) -> pd.Series:
        known_rows = well.known_rows
        prediction_rows = well.prediction_rows
        last_row = well.last_known_row
        slope = self._slope(known_rows)
        md_since = prediction_rows["MD"] - float(last_row["MD"])
        values = float(last_row["TVT_input"]) + slope * md_since
        return pd.Series(values.to_numpy(dtype=float), index=prediction_rows.index, name=self.name)


def rmse(y_true: pd.Series, y_pred: pd.Series) -> float:
    """Computes root mean squared error for aligned numeric arrays."""

    return float(np.sqrt(np.mean((y_true.to_numpy(dtype=float) - y_pred.to_numpy(dtype=float)) ** 2)))


def evaluate_train_wells(loader: WellLoader, predictors: list, max_wells: int | None = None) -> pd.DataFrame:
    """Evaluates baselines on train wells using the hidden TVT_input zone as validation rows."""

    rows = []
    well_ids = loader.well_ids("train")
    if max_wells is not None:
        well_ids = well_ids[:max_wells]

    for well_id in well_ids:
        well = loader.load(well_id, "train")
        truth = well.prediction_rows["TVT"]
        for predictor in predictors:
            prediction = predictor.predict(well)
            rows.append({
                "well_id": well_id,
                "predictor": predictor.name,
                "rows": len(truth),
                "rmse": rmse(truth, prediction),
            })
    return pd.DataFrame(rows)


baseline_predictors = [
    LastKnownBaseline(),
    LinearSlopeBaseline(tail_rows=80),
]

baseline_preview = evaluate_train_wells(loader, baseline_predictors, max_wells=20)
baseline_summary = (
    baseline_preview
    .groupby("predictor", as_index=False)
    .agg(wells=("well_id", "nunique"), rows=("rows", "sum"), mean_rmse=("rmse", "mean"), median_rmse=("rmse", "median"))
    .sort_values("median_rmse")
)

display(baseline_summary)
display(baseline_preview.sort_values("rmse").head(10))


## Validation folds

### Why this block exists

Rows from the same well are highly correlated. A random row split would leak the trajectory shape from train into validation and make the score look better than it really is.

This block creates deterministic well-level folds. Every well belongs to exactly one validation fold. The same split logic will later be reused for ML models and stacking.

### Current scope

For speed, the demo below evaluates the baselines on a limited number of wells. On Kaggle, set `CV_WELL_LIMIT = None` to audit all train wells.


In [ ]:

class WellFoldSplitter:
    """Creates deterministic well-level folds without depending on sklearn."""

    def __init__(self, n_splits: int = 5, seed: int = 42):
        self.n_splits = n_splits
        self.seed = seed

    def assign(self, well_ids: list[str]) -> pd.DataFrame:
        rng = np.random.default_rng(self.seed)
        shuffled = np.array(sorted(well_ids), dtype=object)
        rng.shuffle(shuffled)
        rows = [
            {"well_id": str(well_id), "fold": int(index % self.n_splits)}
            for index, well_id in enumerate(shuffled)
        ]
        return pd.DataFrame(rows).sort_values("well_id").reset_index(drop=True)

    def summary(self, folds: pd.DataFrame) -> pd.DataFrame:
        return (
            folds
            .groupby("fold", as_index=False)
            .agg(wells=("well_id", "count"))
            .sort_values("fold")
        )


def evaluate_predictors_by_fold(
    loader: WellLoader,
    predictors: list,
    folds: pd.DataFrame,
    max_wells: int | None = None,
) -> pd.DataFrame:
    """Evaluates deterministic predictors and attaches the validation fold for each well."""

    fold_map = dict(zip(folds["well_id"], folds["fold"]))
    well_ids = folds["well_id"].tolist()
    if max_wells is not None:
        well_ids = well_ids[:max_wells]

    rows = []
    for well_id in well_ids:
        well = loader.load(well_id, "train")
        truth = well.prediction_rows["TVT"]
        for predictor in predictors:
            prediction = predictor.predict(well)
            rows.append({
                "well_id": well_id,
                "fold": int(fold_map[well_id]),
                "predictor": predictor.name,
                "rows": len(truth),
                "rmse": rmse(truth, prediction),
            })
    return pd.DataFrame(rows)


CV_WELL_LIMIT = 100

splitter = WellFoldSplitter(n_splits=config.n_splits, seed=config.seed)
train_folds = splitter.assign(loader.well_ids("train"))
folded_baseline_scores = evaluate_predictors_by_fold(
    loader=loader,
    predictors=baseline_predictors,
    folds=train_folds,
    max_wells=CV_WELL_LIMIT,
)
folded_baseline_summary = (
    folded_baseline_scores
    .groupby(["predictor", "fold"], as_index=False)
    .agg(wells=("well_id", "nunique"), rows=("rows", "sum"), mean_rmse=("rmse", "mean"), median_rmse=("rmse", "median"))
    .sort_values(["predictor", "fold"])
)

display(splitter.summary(train_folds))
display(folded_baseline_summary)


## Typewell beam-search baseline

### Why this block exists

The previous baselines ignored `typewell`. That is too weak for this competition because `typewell` gives a vertical `TVT -> GR` reference curve.

This block adds a small beam-search tracker. It walks through the prediction zone and chooses a TVT path whose expected `GR` from the typewell is close to the horizontal-well `GR`, while penalizing jumps between neighboring TVT positions.

### Current scope

This is intentionally simple and CPU-friendly. It gives us a first physical candidate that later ML models can use as a feature.


In [ ]:

class TypewellGrid:
    """Interpolates typewell GR values on a regular TVT grid."""

    def __init__(self, typewell: pd.DataFrame, step: float = 1.0):
        clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
        if len(clean) < 3:
            raise ValueError("Typewell must contain at least three non-missing TVT/GR rows")
        self.tvt_min = float(clean["TVT"].min())
        self.tvt_max = float(clean["TVT"].max())
        self.step = float(step)
        self.tvt = np.arange(self.tvt_min, self.tvt_max + self.step, self.step, dtype=float)
        self.gr = np.interp(self.tvt, clean["TVT"].to_numpy(dtype=float), clean["GR"].to_numpy(dtype=float))

    def nearest_index(self, tvt_value: float) -> int:
        return int(np.clip(np.searchsorted(self.tvt, tvt_value), 0, len(self.tvt) - 1))

    def tvt_at(self, index: int) -> float:
        return float(self.tvt[int(np.clip(index, 0, len(self.tvt) - 1))])


class BeamSearchBaseline:
    """Tracks a TVT path by matching horizontal GR to typewell GR."""

    name = "beam_search"

    def __init__(self, grid_step: float = 1.0, beam_width: int = 12, move_radius: int = 2, move_penalty: float = 8.0, gr_scale: float = 80.0):
        self.grid_step = grid_step
        self.beam_width = beam_width
        self.move_radius = move_radius
        self.move_penalty = move_penalty
        self.gr_scale = gr_scale

    def _prediction_gr(self, well: WellData) -> np.ndarray:
        horizontal_gr = well.horizontal["GR"].astype(float).interpolate(limit_direction="both")
        fallback = float(well.typewell["GR"].dropna().mean())
        return horizontal_gr.fillna(fallback).loc[well.prediction_rows.index].to_numpy(dtype=float)

    def predict(self, well: WellData) -> pd.Series:
        grid = TypewellGrid(well.typewell, step=self.grid_step)
        observed_gr = self._prediction_gr(well)
        if len(observed_gr) == 0:
            return pd.Series(dtype=float, name=self.name)

        start_index = grid.nearest_index(float(well.last_known_row["TVT_input"]))
        beams = [(0.0, start_index, [])]
        moves = np.arange(-self.move_radius, self.move_radius + 1, dtype=int)

        for gr_value in observed_gr:
            candidates = []
            for cost, index, path in beams:
                next_indices = np.clip(index + moves, 0, len(grid.tvt) - 1)
                for next_index, move in zip(next_indices, moves):
                    gr_error = (gr_value - grid.gr[next_index]) ** 2 / self.gr_scale
                    move_cost = self.move_penalty * abs(int(move))
                    candidates.append((cost + gr_error + move_cost, int(next_index), path + [int(next_index)]))
            candidates.sort(key=lambda item: item[0])
            beams = candidates[:self.beam_width]

        best_path = beams[0][2]
        values = [grid.tvt_at(index) for index in best_path]
        return pd.Series(values, index=well.prediction_rows.index, name=self.name)


beam_predictor = BeamSearchBaseline()
beam_preview = evaluate_train_wells(loader, [beam_predictor], max_wells=20)
beam_summary = (
    beam_preview
    .groupby("predictor", as_index=False)
    .agg(wells=("well_id", "nunique"), rows=("rows", "sum"), mean_rmse=("rmse", "mean"), median_rmse=("rmse", "median"))
)

display(beam_summary)
display(beam_preview.sort_values("rmse").head(10))


## Candidate feature table

### Why this block exists

ML models should not start from raw CSV rows. They need one clean table where each row is one prediction-zone point and each column has a clear meaning.

This block builds that table. It keeps simple trajectory features and adds every deterministic baseline as both an absolute TVT candidate and a delta from `last_known_tvt`.

### Target definition

For train rows, the model target is:

`target_delta = TVT - last_known_tvt`

The final prediction will later be:

`predicted_tvt = last_known_tvt + predicted_delta`


In [ ]:

class CandidateFeatureBuilder:
    """Builds row-level features for the prediction zone of each well."""

    def __init__(self, predictors: list):
        self.predictors = predictors

    def build_well(self, well: WellData) -> pd.DataFrame:
        prediction_rows = well.prediction_rows
        last_row = well.last_known_row
        last_known_tvt = float(last_row["TVT_input"])
        last_known_md = float(last_row["MD"])
        row_index = prediction_rows.index.to_numpy(dtype=int)
        frame = pd.DataFrame({
            "well_id": well.well_id,
            "id": [f"{well.well_id}_{index}" for index in row_index],
            "row_index": row_index,
            "md": prediction_rows["MD"].to_numpy(dtype=float),
            "x": prediction_rows["X"].to_numpy(dtype=float),
            "y": prediction_rows["Y"].to_numpy(dtype=float),
            "z": prediction_rows["Z"].to_numpy(dtype=float),
            "gr": prediction_rows["GR"].to_numpy(dtype=float),
            "gr_missing": prediction_rows["GR"].isna().astype(int).to_numpy(),
            "last_known_tvt": last_known_tvt,
            "last_known_md": last_known_md,
            "md_since": prediction_rows["MD"].to_numpy(dtype=float) - last_known_md,
            "known_rows": int(well.known_mask.sum()),
            "prediction_rows_total": int(well.prediction_mask.sum()),
        })
        frame["frac"] = np.arange(len(frame), dtype=float) / max(len(frame) - 1, 1)

        for predictor in self.predictors:
            prediction = predictor.predict(well).reindex(prediction_rows.index)
            frame[f"{predictor.name}_tvt"] = prediction.to_numpy(dtype=float)
            frame[f"{predictor.name}_delta"] = frame[f"{predictor.name}_tvt"] - last_known_tvt

        if "TVT" in prediction_rows.columns:
            frame["target_tvt"] = prediction_rows["TVT"].to_numpy(dtype=float)
            frame["target_delta"] = frame["target_tvt"] - last_known_tvt

        return frame

    def build(self, loader: WellLoader, split: str, well_ids: list[str]) -> pd.DataFrame:
        parts = [self.build_well(loader.load(well_id, split)) for well_id in well_ids]
        if not parts:
            return pd.DataFrame()
        return pd.concat(parts, ignore_index=True)

    def feature_columns(self, frame: pd.DataFrame) -> list[str]:
        blocked = {"well_id", "id", "target_tvt", "target_delta"}
        return [column for column in frame.columns if column not in blocked]


candidate_predictors = [
    LastKnownBaseline(),
    LinearSlopeBaseline(tail_rows=80),
    BeamSearchBaseline(),
]
feature_builder = CandidateFeatureBuilder(candidate_predictors)

FEATURE_PREVIEW_WELLS = 5
train_feature_preview = feature_builder.build(loader, "train", loader.well_ids("train")[:FEATURE_PREVIEW_WELLS])
test_feature_preview = feature_builder.build(loader, "test", loader.well_ids("test"))
feature_columns = feature_builder.feature_columns(train_feature_preview)

print(f"train preview shape: {train_feature_preview.shape}")
print(f"test preview shape: {test_feature_preview.shape}")
print(f"feature columns: {len(feature_columns)}")

display(train_feature_preview.head())
display(pd.DataFrame({"feature": feature_columns}).head(30))


## Sequence feature upgrade

### Why this block exists

The first feature table is intentionally plain. This block adds local sequence context from the horizontal well: rolling GR statistics, short-window slopes, and coordinate movement. These features let a tabular model see curve shape without building a full neural sequence model yet.

In [ ]:
class SequenceFeatureAugmenter:
    """Adds local sequence features inside each well."""

    def __init__(self, windows: tuple[int, ...] = (5, 25, 75)):
        self.windows = windows

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        result = frame.copy()
        result = result.sort_values(["well_id", "row_index"]).reset_index(drop=True)
        grouped = result.groupby("well_id", sort=False)

        result["md_step"] = grouped["md"].diff().fillna(0.0)
        result["gr_step"] = grouped["gr"].diff().fillna(0.0)
        result["z_step"] = grouped["z"].diff().fillna(0.0)
        result["xy_step"] = np.sqrt(grouped["x"].diff().fillna(0.0) ** 2 + grouped["y"].diff().fillna(0.0) ** 2)

        for window in self.windows:
            rolling_gr = grouped["gr"].rolling(window=window, min_periods=1)
            result[f"gr_roll_mean_{window}"] = rolling_gr.mean().reset_index(level=0, drop=True)
            result[f"gr_roll_std_{window}"] = rolling_gr.std().reset_index(level=0, drop=True).fillna(0.0)
            result[f"gr_roll_min_{window}"] = rolling_gr.min().reset_index(level=0, drop=True)
            result[f"gr_roll_max_{window}"] = rolling_gr.max().reset_index(level=0, drop=True)

        numeric_columns = result.select_dtypes(include=[np.number]).columns
        result[numeric_columns] = result[numeric_columns].replace([np.inf, -np.inf], np.nan)
        result[numeric_columns] = result[numeric_columns].fillna(result[numeric_columns].median(numeric_only=True))
        result[numeric_columns] = result[numeric_columns].fillna(0.0)
        return result


sequence_augmenter = SequenceFeatureAugmenter()
train_feature_preview_augmented = sequence_augmenter.transform(train_feature_preview)
test_feature_preview_augmented = sequence_augmenter.transform(test_feature_preview)
augmented_feature_columns = feature_builder.feature_columns(train_feature_preview_augmented)

print(f"augmented train preview shape: {train_feature_preview_augmented.shape}")
print(f"augmented test preview shape: {test_feature_preview_augmented.shape}")
print(f"augmented feature columns: {len(augmented_feature_columns)}")

display(pd.DataFrame({"feature": augmented_feature_columns}).tail(20))

## Gradient boosting model

### Why this block exists

This block trains the first supervised model on top of the candidate feature table. The model predicts `target_delta`, not raw `TVT`, because the last known `TVT` already gives a strong local anchor.

### Practical choice

On Kaggle, the notebook will try `HistGradientBoostingRegressor` if scikit-learn is available. If it is not available, the same interface falls back to a small NumPy ridge model so the notebook still runs end to end.

In [ ]:
class RidgeDeltaRegressor:
    """Small closed-form ridge regressor with median imputation and standardization."""

    name = "ridge_delta"

    def __init__(self, alpha: float = 10.0):
        self.alpha = alpha
        self.columns: list[str] = []
        self.medians: pd.Series | None = None
        self.means: np.ndarray | None = None
        self.scales: np.ndarray | None = None
        self.weights: np.ndarray | None = None

    def fit(self, frame: pd.DataFrame, feature_columns: list[str], target_column: str = "target_delta") -> "RidgeDeltaRegressor":
        self.columns = list(feature_columns)
        self.medians = frame[self.columns].median(numeric_only=True).fillna(0.0)
        x = frame[self.columns].fillna(self.medians).to_numpy(dtype=float)
        y = frame[target_column].to_numpy(dtype=float)
        self.means = x.mean(axis=0)
        self.scales = x.std(axis=0)
        self.scales[self.scales == 0.0] = 1.0
        x_scaled = (x - self.means) / self.scales
        x_design = np.column_stack([np.ones(len(x_scaled)), x_scaled])
        penalty = np.eye(x_design.shape[1]) * self.alpha
        penalty[0, 0] = 0.0
        self.weights = np.linalg.solve(x_design.T @ x_design + penalty, x_design.T @ y)
        return self

    def predict_delta(self, frame: pd.DataFrame) -> np.ndarray:
        if self.medians is None or self.means is None or self.scales is None or self.weights is None:
            raise ValueError("Model is not fitted")
        x = frame[self.columns].fillna(self.medians).to_numpy(dtype=float)
        x_scaled = (x - self.means) / self.scales
        x_design = np.column_stack([np.ones(len(x_scaled)), x_scaled])
        return x_design @ self.weights

    def predict_tvt(self, frame: pd.DataFrame) -> np.ndarray:
        return frame["last_known_tvt"].to_numpy(dtype=float) + self.predict_delta(frame)


class SklearnGradientBoostingDeltaRegressor:
    """Wrapper around sklearn HistGradientBoostingRegressor when it is available."""

    name = "hist_gradient_boosting_delta"

    def __init__(self, random_state: int = 42):
        from sklearn.ensemble import HistGradientBoostingRegressor
        self.model = HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=500,
            l2_regularization=0.05,
            random_state=random_state,
        )
        self.columns: list[str] = []
        self.medians: pd.Series | None = None

    def fit(self, frame: pd.DataFrame, feature_columns: list[str], target_column: str = "target_delta") -> "SklearnGradientBoostingDeltaRegressor":
        self.columns = list(feature_columns)
        self.medians = frame[self.columns].median(numeric_only=True).fillna(0.0)
        x = frame[self.columns].fillna(self.medians)
        y = frame[target_column].to_numpy(dtype=float)
        self.model.fit(x, y)
        return self

    def predict_delta(self, frame: pd.DataFrame) -> np.ndarray:
        if self.medians is None:
            raise ValueError("Model is not fitted")
        return self.model.predict(frame[self.columns].fillna(self.medians))

    def predict_tvt(self, frame: pd.DataFrame) -> np.ndarray:
        return frame["last_known_tvt"].to_numpy(dtype=float) + self.predict_delta(frame)


def make_delta_model(seed: int = 42):
    """Uses sklearn boosting when available and falls back to NumPy ridge otherwise."""

    try:
        return SklearnGradientBoostingDeltaRegressor(random_state=seed)
    except Exception:
        return RidgeDeltaRegressor(alpha=10.0)


model_preview = make_delta_model(seed=config.seed)
model_preview.fit(train_feature_preview_augmented, augmented_feature_columns)
preview_prediction = model_preview.predict_tvt(train_feature_preview_augmented)
preview_rmse = rmse(train_feature_preview_augmented["target_tvt"], pd.Series(preview_prediction))

print(f"model: {model_preview.name}")
print(f"preview train rmse: {preview_rmse:.5f}")

## Validation report

### Why this block exists

This block evaluates the supervised model on well-level folds. Rows from the same well must stay together, otherwise the validation score leaks sequence-specific information and becomes too optimistic.

In [ ]:
model_feature_builder = CandidateFeatureBuilder([
    LastKnownBaseline(),
    LinearSlopeBaseline(tail_rows=80),
])


def build_training_features_for_wells(well_ids: list[str]) -> pd.DataFrame:
    """Builds fast augmented training features for a list of train wells."""

    base = model_feature_builder.build(loader, "train", well_ids)
    return sequence_augmenter.transform(base)


def evaluate_delta_model_by_fold(folds: pd.DataFrame, max_wells: int | None = None) -> pd.DataFrame:
    """Trains one model per fold and reports RMSE on held-out wells."""

    selected_folds = folds.copy()
    if max_wells is not None:
        selected_folds = selected_folds.head(max_wells)

    rows = []
    for fold in sorted(selected_folds["fold"].unique()):
        train_ids = selected_folds.loc[selected_folds["fold"] != fold, "well_id"].tolist()
        valid_ids = selected_folds.loc[selected_folds["fold"] == fold, "well_id"].tolist()
        train_frame = build_training_features_for_wells(train_ids)
        valid_frame = build_training_features_for_wells(valid_ids)
        columns = model_feature_builder.feature_columns(train_frame)
        model = make_delta_model(seed=config.seed + int(fold))
        model.fit(train_frame, columns)
        prediction = model.predict_tvt(valid_frame)
        rows.append({
            "fold": int(fold),
            "model": model.name,
            "train_wells": len(train_ids),
            "valid_wells": len(valid_ids),
            "valid_rows": len(valid_frame),
            "rmse": rmse(valid_frame["target_tvt"], pd.Series(prediction)),
        })
    return pd.DataFrame(rows)


LOCAL_VALIDATION_WELLS = 20
KAGGLE_VALIDATION_WELLS = 160
validation_well_limit = KAGGLE_VALIDATION_WELLS if Path("/kaggle/input").exists() else LOCAL_VALIDATION_WELLS
model_validation = evaluate_delta_model_by_fold(train_folds, max_wells=validation_well_limit)

print(f"validation wells used: {validation_well_limit}")
display(model_validation)
display(pd.DataFrame([{
    "rmse_mean": float(model_validation["rmse"].mean()),
    "rmse_median": float(model_validation["rmse"].median()),
}]))

## Full train and submission

### Why this block exists

This block trains the final model and writes `submission.csv`. Locally it uses a bounded number of wells so the notebook stays quick. On Kaggle it uses more wells by default and writes to `/kaggle/working`.

In [ ]:
LOCAL_TRAIN_WELLS = 80
KAGGLE_TRAIN_WELLS = None
train_well_limit = KAGGLE_TRAIN_WELLS if Path("/kaggle/input").exists() else LOCAL_TRAIN_WELLS
train_well_ids = loader.well_ids("train") if train_well_limit is None else loader.well_ids("train")[:train_well_limit]

final_train_features = build_training_features_for_wells(train_well_ids)
final_test_features = sequence_augmenter.transform(feature_builder.build(loader, "test", loader.well_ids("test")))
final_feature_columns = model_feature_builder.feature_columns(final_train_features)

final_model = make_delta_model(seed=config.seed)
final_model.fit(final_train_features, final_feature_columns)
final_test_features["model_tvt"] = final_model.predict_tvt(final_test_features)

submission = pd.read_csv(config.sample_submission_path)
submission_values = final_test_features[["id", "model_tvt"]].rename(columns={"model_tvt": "tvt"})
submission = submission[["id"]].merge(submission_values, on="id", how="left")
if submission["tvt"].isna().any():
    raise ValueError("Submission contains missing predictions")

submission_path = config.work_dir / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"train wells used: {len(train_well_ids)}")
print(f"submission rows: {len(submission)}")
print(f"submission path: {submission_path}")
display(submission.head())

## Model diagnostics

### Why this block exists

This block checks where the model is weak. The fastest useful diagnostic is well-level error: it tells us whether the model improves broadly or only on easy wells.

In [ ]:
def diagnostic_holdout_report(well_ids: list[str]) -> pd.DataFrame:
    """Compares model and baseline errors on a small holdout slice."""

    if len(well_ids) < 4:
        return pd.DataFrame()
    split_at = max(1, int(len(well_ids) * 0.75))
    train_ids = well_ids[:split_at]
    valid_ids = well_ids[split_at:]
    train_frame = build_training_features_for_wells(train_ids)
    valid_frame = build_training_features_for_wells(valid_ids)
    columns = model_feature_builder.feature_columns(train_frame)
    model = make_delta_model(seed=config.seed)
    model.fit(train_frame, columns)
    valid_frame = valid_frame.copy()
    valid_frame["model_tvt"] = model.predict_tvt(valid_frame)
    rows = []
    for well_id, group in valid_frame.groupby("well_id"):
        rows.append({
            "well_id": well_id,
            "rows": len(group),
            "last_known_rmse": rmse(group["target_tvt"], group["last_known_tvt"]),
            "linear_slope_rmse": rmse(group["target_tvt"], group["linear_slope_tvt"]),
            "model_rmse": rmse(group["target_tvt"], group["model_tvt"]),
        })
    result = pd.DataFrame(rows)
    result["model_vs_last_known"] = result["model_rmse"] - result["last_known_rmse"]
    return result.sort_values("model_rmse")


diagnostic_wells = loader.well_ids("train")[:max(validation_well_limit, 20)]
diagnostic_report = diagnostic_holdout_report(diagnostic_wells)

display(diagnostic_report.head(10))
display(diagnostic_report.tail(10))
display(diagnostic_report[["last_known_rmse", "linear_slope_rmse", "model_rmse", "model_vs_last_known"]].mean().to_frame("mean_value"))

## Ensemble and post-processing

### Why this block exists

This block adds a conservative blend and monotonic smoothing. It is deliberately small: the blend keeps the supervised model anchored to the strong last-known baseline, and smoothing removes obvious local jumps.

In [ ]:
class ConservativeEnsembler:
    """Blends model predictions with simple physical anchors."""

    def __init__(self, model_weight: float = 0.70, beam_weight: float = 0.20, last_known_weight: float = 0.10):
        total = model_weight + beam_weight + last_known_weight
        self.model_weight = model_weight / total
        self.beam_weight = beam_weight / total
        self.last_known_weight = last_known_weight / total

    def blend(self, frame: pd.DataFrame, model_column: str = "model_tvt") -> np.ndarray:
        return (
            self.model_weight * frame[model_column].to_numpy(dtype=float)
            + self.beam_weight * frame["beam_search_tvt"].to_numpy(dtype=float)
            + self.last_known_weight * frame["last_known_tvt"].to_numpy(dtype=float)
        )


class WellPostProcessor:
    """Applies light per-well smoothing to TVT predictions."""

    def __init__(self, max_step: float = 2.5):
        self.max_step = max_step

    def transform(self, frame: pd.DataFrame, value_column: str) -> np.ndarray:
        result = np.empty(len(frame), dtype=float)
        for _, group in frame.groupby("well_id", sort=False):
            values = group[value_column].to_numpy(dtype=float).copy()
            for index in range(1, len(values)):
                step = np.clip(values[index] - values[index - 1], -self.max_step, self.max_step)
                values[index] = values[index - 1] + step
            result[group.index.to_numpy()] = values
        return result


ensembler = ConservativeEnsembler()
postprocessor = WellPostProcessor(max_step=2.5)

final_test_features["ensemble_tvt"] = ensembler.blend(final_test_features)
final_test_features["postprocessed_tvt"] = postprocessor.transform(final_test_features, "ensemble_tvt")

final_submission = pd.read_csv(config.sample_submission_path)[["id"]].merge(
    final_test_features[["id", "postprocessed_tvt"]].rename(columns={"postprocessed_tvt": "tvt"}),
    on="id",
    how="left",
)
if final_submission["tvt"].isna().any():
    raise ValueError("Final submission contains missing predictions")

final_submission_path = config.work_dir / "submission_postprocessed.csv"
final_submission.to_csv(final_submission_path, index=False)

print(f"final submission path: {final_submission_path}")
display(final_submission.head())
display(final_submission.tail())